# Outlines-constrained actions with native smolagents

This synthetic example chooses the lowest-cost box that fits a padded device requiring **200 × 140 × 60 mm** of interior space. Compact has a **198 × 145 × 65 mm** interior and costs **USD 5.45** delivered. Standard has a **210 × 150 × 70 mm** interior and costs **USD 6.35** delivered. The budget is **USD 6.50**.

`OutlinesToolModel` reads the schemas produced by smolagents' `@tool` decorator. `Literal["Compact", "Standard"]` becomes an allowed-value list for each tool argument. Outlines constrains each generated action, and the model validates it before the native agent executes it. The tools return short Python strings as observations. Calls can repeat or occur in any order; the schema does not judge the final choice.

The final-answer schema below requires a box name from the two choices and a nonnegative numeric delivered price. It constrains the fields and types, while the agent uses observations to determine their values. Omit `final_answer_schema` for a free-form string, or use `final_answers=[...]` for an enumerated string. The same model can be passed to native `ToolCallingAgent` or `CodeAgent` with the same tools list.

Run top to bottom in Colab. The first run downloads public Qwen2.5-1.5B-Instruct weights; a T4 GPU is helpful. No API key is needed.


In [ ]:
%pip install -q "git+https://github.com/cmccomb/smolagents-outlines.git"


In [ ]:
from typing import Literal
from smolagents import tool

REQUIRED_MM = (200, 140, 60)
BOXES = {
    "Compact": {"inner_mm": (198, 145, 65), "delivered_usd": 5.45},
    "Standard": {"inner_mm": (210, 150, 70), "delivered_usd": 6.35},
}
BUDGET_USD = 6.50

@tool
def check_fit(box: Literal["Compact", "Standard"]) -> str:
    """Check whether a box fits the padded device.

    Args:
        box: Box to check.
    """
    fits = all(inner >= required for inner, required in zip(BOXES[box]["inner_mm"], REQUIRED_MM))
    return f"{box} {'fits' if fits else 'does not fit'} the padded device."

@tool
def delivery_cost(box: Literal["Compact", "Standard"]) -> str:
    """Report the delivered cost and budget result for a box.

    Args:
        box: Box to price.
    """
    cost = BOXES[box]["delivered_usd"]
    return f"{box} costs ${cost:.2f} delivered; {'within' if cost <= BUDGET_USD else 'over'} budget."

TOOLS = [check_fit, delivery_cost]
assert all(t.inputs["box"]["enum"] == list(BOXES) for t in TOOLS)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from smolagents import CodeAgent, ToolCallingAgent
from smolagents_outlines import OutlinesToolModel

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device in ("cuda", "mps") else torch.float32

# Public weights: avoid Colab's implicit HF_TOKEN secret prompt.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=False)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map=device, dtype=dtype, token=False
)
FINAL_ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "selected_box": {"type": "string", "enum": list(BOXES)},
        "delivered_price_usd": {"type": "number", "minimum": 0},
    },
    "required": ["selected_box", "delivered_price_usd"],
    "additionalProperties": False,
}
model = OutlinesToolModel(
    hf_model, tokenizer, TOOLS, final_answer_schema=FINAL_ANSWER_SCHEMA
)
AGENT_CLASS = ToolCallingAgent  # Change to CodeAgent to use its native execution path.
agent = AGENT_CLASS(tools=TOOLS, model=model, max_steps=8)


In [ ]:
answer = agent.run(
    "Choose the lowest-cost box that fits the padded device and stays within the $6.50 budget. In your final answer, give the box name and delivered price."
)
print("Answer:", answer)
